# Part 4.1 | Mature DPD30 by month and acquisition channel

Run the prep cell and then the SQL cell to reproduce the output from the original CSVs. The prep cell registers a local SQLite-backed SQL magic that translates the date syntax. Copy the SQL body directly into a PostgreSQL or Redshift client with the `lucky_case` schema.

The source extract has unique loan IDs and user IDs, and all loan borrowers match a user. The user CTE collapses repeated copies of the same normalized source. Conflicting sources for the same user ID would need review before joining. The channel mapping follows the reporting notebooks.

The denominator includes only loans that reached DPD30 by November 5, 2026. The query uses loan dates as supplied and groups by disbursal month. For fully repaid loans, the last recorded payment date is used as a proxy for when repayment was completed. Full repayment is principal plus fee, with a one-piastre tolerance.

In [7]:
from pathlib import Path
import re
import sqlite3

import pandas as pd
from IPython import get_ipython
from IPython.display import display

ROOT = next(
    folder for folder in (Path.cwd(), Path.cwd().parent)
    if (folder / "data" / "raw").is_dir()
)
RAW = ROOT / "data" / "raw"
users = pd.read_csv(next(RAW.glob("* - users.csv")))
loans = pd.read_csv(next(RAW.glob("* - loans.csv")))

# Check the source grain before joining. Exact user copies can be collapsed.
assert users.drop_duplicates()["user_id"].is_unique
assert loans["loan_id"].notna().all() and loans["loan_id"].is_unique
assert loans["user_id"].isin(users["user_id"]).all()

connection = sqlite3.connect(":memory:")
connection.execute("ATTACH DATABASE ':memory:' AS lucky_case")
users.to_sql("raw_users", connection, index=False)
loans.to_sql("raw_loans", connection, index=False)
connection.execute("CREATE TABLE lucky_case.users AS SELECT * FROM raw_users")
connection.execute("CREATE TABLE lucky_case.loans AS SELECT * FROM raw_loans")


def run_case_sql(line, cell):
    if line.strip():
        raise ValueError("This %%sql cell takes no arguments.")
    # Translate only the date expressions SQLite spells differently.
    local_sql = re.sub(r"CAST\((l\.\w+) AS DATE\)", r"DATE(\1)", cell)
    local_sql = local_sql.replace(
        "DATE_TRUNC('month', DATE(l.disbursed_date))",
        "DATE(l.disbursed_date, 'start of month')",
    )
    local_sql = local_sql.replace(
        "DATE(l.due_date) + INTERVAL '30 days'", "DATE(l.due_date, '+30 days')"
    )
    local_sql = re.sub(r"\bDATE ('\d{4}-\d{2}-\d{2}')", r"\1", local_sql)
    result = pd.read_sql_query(local_sql, connection)
    display(result)


get_ipython().register_magic_function(run_case_sql, magic_kind="cell", magic_name="sql")

In [8]:
%%sql
-- Part 4.1: mature DPD30 by disbursal month and acquisition channel.
WITH clean_users AS (
    SELECT DISTINCT user_id, LOWER(TRIM(utm_source)) AS source
    FROM lucky_case.users
    WHERE user_id IS NOT NULL
      AND COALESCE(LOWER(TRIM(full_name)), '') NOT IN (
          'qa lucky', 'test user', 'test qa', 'test test',
          'تجربه تطبيق', 'تجربة'
      )
),
user_channels AS (
    SELECT user_id,
           CASE
               WHEN source IN ('meta', 'facebook ads', 'facebook', 'fb', 'instagram', 'ig') THEN 'Meta'
               WHEN source IN ('google', 'google ads', 'adwords', 'gads') THEN 'Google Ads'
               WHEN source IN ('tiktok', 'tt') THEN 'TikTok'
               WHEN source IN ('organic', '(direct)', 'referral', 'ref', 'invite') THEN 'Other'
               ELSE 'Unknown'
           END AS channel
    FROM clean_users
),
loan_checkpoints AS (
    SELECT l.loan_id,
           DATE_TRUNC('month', CAST(l.disbursed_date AS DATE)) AS disbursal_month,
           u.channel,
           CAST(l.due_date AS DATE) + INTERVAL '30 days' AS dpd30_date,
           l.principal_egp + l.fee_egp AS amount_due,
           l.amount_repaid_egp,
           CAST(l.last_repayment_date AS DATE) AS last_repayment_date
    FROM lucky_case.loans AS l
    JOIN user_channels AS u ON u.user_id = l.user_id
    WHERE CAST(l.disbursed_date AS DATE) >= DATE '2026-07-01'
      AND CAST(l.disbursed_date AS DATE) < DATE '2026-10-01'
),
mature_loans AS (
    -- A final payment on the checkpoint counts as repayment by DPD30.
    SELECT disbursal_month, channel,
           CASE WHEN amount_repaid_egp >= amount_due - 0.01
                     AND last_repayment_date <= dpd30_date
                THEN 0 ELSE 1 END AS dpd30_default
    FROM loan_checkpoints
    WHERE dpd30_date <= DATE '2026-11-05'
)
SELECT disbursal_month, channel,
       COUNT(*) AS mature_loans,
       SUM(dpd30_default) AS dpd30_defaults,
       ROUND(100.0 * SUM(dpd30_default) / COUNT(*), 2) AS dpd30_rate_pct
FROM mature_loans
GROUP BY disbursal_month, channel
ORDER BY disbursal_month, channel;

,disbursal_month,channel,mature_loans,dpd30_defaults,dpd30_rate_pct
0,2026-07-01,Google Ads,81,2,2.47
1,2026-07-01,Meta,52,5,9.62
2,2026-07-01,Other,80,6,7.50
3,2026-07-01,TikTok,16,3,18.75
4,2026-07-01,Unknown,31,1,3.23
5,2026-08-01,Google Ads,142,5,3.52
6,2026-08-01,Meta,107,16,14.95
7,2026-08-01,Other,110,2,1.82
8,2026-08-01,TikTok,21,0,0.00
9,2026-08-01,Unknown,47,2,4.26


The channel counts roll up to **17/260 in July**, **25/427 in August**, and **34/353 in September**.